# Training notebook overview
This notebook takes the preprocessed data and trains Random Forest models: a baseline with 10-fold CV, a tuned model via RandomizedSearchCV, and an EDA-informed model using top mutual-information features. Artifacts (models, plots, CV summaries) are saved under `../result/`.

In [1]:
# Hyperparameter tuning with cross-validation
from utils import plot_feature_importances
from sklearn.model_selection import RandomizedSearchCV , StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from scipy.stats import randint
import numpy as np
import pandas as pd
import joblib
import os

SEED = 42
PROCESS_PATH = '../result/processed'
MODEL_PATH = '../result/model'
PIC_PATH = '../result/pic'
CV_PATH = '../result/cv'
# Use the already prepared train_final (one-hot)
train_final = pd.read_csv(f'{PROCESS_PATH}/titanic_train_preprocessed.csv')
X = train_final.drop(columns=['Survived', 'PassengerId'])
y = train_final['Survived']


In [2]:

kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

accs, aucs = [], []

for fold_idx, (train_idx, valid_idx) in enumerate(kfold.split(X, y), start=1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    rf = RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features='sqrt',
        class_weight='balanced',
        n_jobs=-1,
        random_state=SEED,
    )
    rf.fit(X_train, y_train)

    pred = rf.predict(X_valid)
    proba = rf.predict_proba(X_valid)[:, 1]

    acc = accuracy_score(y_valid, pred)
    auc = roc_auc_score(y_valid, proba)
    accs.append(acc)
    aucs.append(auc)
    print(f'Fold {fold_idx}: accuracy={acc:.4f}, roc_auc={auc:.4f}')

print('\nCV summary (10-fold):')
print({'accuracy_mean': round(np.mean(accs), 4), 'accuracy_std': round(np.std(accs), 4),
       'roc_auc_mean': round(np.mean(aucs), 4), 'roc_auc_std': round(np.std(aucs), 4)})

# Refit on full data for deployment and save
Base_rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=SEED,
)
Base_rf.fit(X, y)

plot_feature_importances(Base_rf, X.columns, top_n=20, fname=f'{PIC_PATH}/rf_feature_importances_top20.png')
import time

timestamp = time.strftime("%Y%m%d-%H%M%S")
os.makedirs(PROCESS_PATH, exist_ok=True)
joblib.dump({'model': Base_rf, 'features': X.columns.tolist()}, f'{MODEL_PATH}/randomForest_Base_{timestamp}.pkl')
print(f'Saved baseline model to {MODEL_PATH}/randomForest_Base_{timestamp}.pkl')

Fold 1: accuracy=0.7778, roc_auc=0.8639
Fold 2: accuracy=0.8315, roc_auc=0.8834
Fold 3: accuracy=0.8315, roc_auc=0.8516
Fold 4: accuracy=0.7640, roc_auc=0.8436
Fold 5: accuracy=0.7865, roc_auc=0.8251
Fold 6: accuracy=0.8090, roc_auc=0.8751
Fold 7: accuracy=0.8539, roc_auc=0.8880
Fold 8: accuracy=0.7865, roc_auc=0.8262
Fold 9: accuracy=0.8202, roc_auc=0.9013
Fold 10: accuracy=0.7865, roc_auc=0.8466

CV summary (10-fold):
{'accuracy_mean': np.float64(0.8047), 'accuracy_std': np.float64(0.0274), 'roc_auc_mean': np.float64(0.8605), 'roc_auc_std': np.float64(0.0248)}
Saved baseline model to ../result/model/randomForest_Base_20251102-125843.pkl


## Baseline Random Forest with 10-fold CV, then refit full and save

This cell establishes a strong baseline and produces a fully trained, reproducible model artifact. It begins by loading the one‑hot encoded training matrix produced in `code.ipynb` (which already handled imputation, feature creation such as Title/FamilySize/IsAlone, and categorical encoding). We separate the target `y = Survived` and features `X` (dropping `Survived` and `PassengerId`), ensuring there’s no label leakage into the inputs. We use `StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)` to obtain reliable estimates of generalization performance under roughly balanced class proportions across folds. For each fold, we fit a `RandomForestClassifier` with a reasonable, defensible set of hyperparameters: 400 trees for stability; `max_features='sqrt'` as a common default for classification forests; and `class_weight='balanced'` to counter Titanic’s moderate class imbalance. We capture two complementary metrics: accuracy (thresholded performance) and ROC AUC (ranking quality independent of threshold).

After iterating through folds, we summarize mean and standard deviation for both metrics. This gives you signal on both central tendency and variance (i.e., stability) of the model. Next, we refit the same Random Forest on the entire dataset `(X, y)` to exploit all labeled data when creating the deployable artifact. We also generate and persist a top‑N feature importance plot to `PIC_PATH` to make the model more interpretable and to inform future feature selection. Finally, we save a versioned pickle to `MODEL_PATH` that includes both the fitted estimator and the exact feature list used during training. Embedding the feature list is crucial: it allows downstream inference (in `model.ipynb`) to align columns precisely, avoiding silent mismatches that degrade predictions. The `SEED` and `timestamp` make runs reproducible and traceable across notebooks.

In [3]:
param_dist = {
    'n_estimators': randint(300, 900),
    'max_depth': [None] + list(range(4, 13)),
    'min_samples_split': randint(2, 11),
    'min_samples_leaf': randint(1, 5),
    'max_features': ['sqrt','log2', 0.5, 0.7],
    'bootstrap': [True],
    'class_weight': ['balanced', None],
    'oob_score': [True, False],
}


randomForest_tuner = RandomForestClassifier(n_jobs=-1, random_state=SEED)
randomSearchCV = RandomizedSearchCV(
    randomForest_tuner , param_distributions=param_dist, n_iter=100,
    scoring='roc_auc', cv = StratifiedKFold(n_splits=10), random_state=SEED, n_jobs=-1, verbose=1, refit=True
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
randomSearchCV.fit(X_train, y_train)

print('score: ', randomSearchCV.cv_results_)

randomForest_best = randomSearchCV.best_estimator_
print('Best params:',  randomSearchCV.best_params_)
print('Best CV ROC AUC:', round(randomSearchCV.best_score_,4))

# Evaluate tuned model
pred2 = randomForest_best.predict(X_valid)
proba2 = randomForest_best.predict_proba(X_valid)[:,1]
acc2 = accuracy_score(y_valid, pred2)
auc2 = roc_auc_score(y_valid, proba2)


print(f"Tuned Random Forest - Accuracy: {acc2:.4f}, ROC AUC: {auc2:.4f}")
# Ensure output directory exists
os.makedirs(CV_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)

plot_feature_importances(randomForest_best, X_train.columns, top_n=20, fname=f'{PIC_PATH}/rf_Best_feature_importances_top20.png')

# Save tuning results with a clearer filename
joblib.dump({'best_params': randomSearchCV.best_params_, 'best_cv_roc_auc': randomSearchCV.best_score_}, f"{CV_PATH}/rf_randomized_search_cv_results_{timestamp}.pkl")
print(f"Saved tuning results to {MODEL_PATH}/rf_randomized_search_cv_results_{timestamp}.pkl")

# Save the best model trained on all features
joblib.dump({'model': randomForest_best, 'features': X.columns.tolist()}, f"{MODEL_PATH}/rf_best_all_features_{timestamp}.pkl")
print(f"Saved tuned model to {MODEL_PATH}/rf_best_all_features_{timestamp}.pkl")

Fitting 10 folds for each of 100 candidates, totalling 1000 fits


score:  {'mean_fit_time': array([0.70710547, 0.95037324, 0.88466113, 0.59903045, 1.24519949,
       1.96620829, 0.74012659, 0.95574386, 1.24707716, 1.03371642,
       0.51311948, 0.60105555, 0.8587908 , 0.65597267, 1.01228819,
       1.38147049, 1.27616405, 0.58513482, 0.57453926, 1.2099462 ,
       0.98076298, 1.06577275, 1.46180186, 0.61883752, 0.37276604,
       0.87490966, 0.85063639, 0.83939812, 1.31155388, 0.84127083,
       0.64487402, 0.77213223, 0.99525638, 0.83709152, 0.92977009,
       1.26373391, 1.09761665, 1.25551322, 0.95954382, 1.35958927,
       0.57209117, 0.75086799, 1.4091459 , 0.89674957, 1.00732591,
       0.67774398, 1.18613207, 1.60178907, 1.09694798, 0.98374875,
       1.59972215, 2.28054535, 1.7198961 , 1.11711009, 0.62108254,
       1.25479727, 0.90655231, 1.70643544, 0.80966463, 1.67505283,
       0.69781675, 0.98343287, 1.07406797, 1.26558859, 0.64191325,
       0.71596351, 0.74419537, 0.53742151, 1.22973888, 0.54281657,
       0.8133512 , 1.14258795, 1.186

## Hyperparameter tuning with RandomizedSearchCV

This cell performs targeted hyperparameter exploration for the Random Forest using `RandomizedSearchCV`, which trades exhaustive coverage for breadth and efficiency. First, we re‑load the full preprocessed training matrix and perform a stratified hold‑out split (80/20) to create a lightweight, unbiased validation set that is separate from the inner CV happening inside the search. Then we define a search space that reflects common RF trade‑offs: `n_estimators` (more trees reduce variance but increase compute), `max_depth` (None lets trees grow fully, while limited depths reduce overfitting), `min_samples_split` and `min_samples_leaf` (regularize leaf size to control complexity), `max_features` (feature subsampling per split for decorrelation), `bootstrap` (typical=True for RF), `class_weight` (address class imbalance), and `oob_score` (out‑of‑bag estimates as an auxiliary diagnostic when bootstrapping).

We run `RandomizedSearchCV` with `n_iter=100` and a 10‑fold `StratifiedKFold` inside the search, optimizing `roc_auc` to be threshold‑invariant. This yields a best estimator that is the result of cross‑validated ranking across sampled hyperparameter combinations. We then evaluate the chosen model only once on the previously held‑out validation set, reporting accuracy and ROC AUC as pragmatic checks against optimistic bias. Next, we persist key artifacts: the best parameter dictionary and cross‑validated AUC to `CV_PATH` for auditability, the tuned model to `MODEL_PATH` coupled with the full feature list, and a feature‑importance bar chart to `PIC_PATH`.

Two practical notes: (1) The hold‑out estimate complements but does not replace nested CV; it’s a fast pragmatic safeguard. (2) If you re‑run the search with different seeds or more iterations, results may improve slightly; however, diminishing returns typically set in. The saved best parameters can be reused later to initialize consistent models without re‑searching, expediting experimentation and deployment.

In [ ]:

# Load mutual information ranking computed in EDA

mi_df = pd.read_csv(f'{PROCESS_PATH}/eda_mutual_information_top30.csv')
mi_df = mi_df.rename(columns={mi_df.columns[0]: 'feature', mi_df.columns[1]: 'mi'})

# Choose top-K features (intersection with training columns to be safe)
TOP_K = 25
topk = mi_df.sort_values('mi', ascending=False).head(TOP_K)['feature'].tolist()
selected_cols = [c for c in topk if c in X.columns]
if len(selected_cols) < max(10, TOP_K//2):
    # Fallback: if few overlap (naming drift), just keep numeric + high-signal basics
    baseline_keep = [c for c in X.columns if any(p in c for p in ['Sex','Pclass','Fare','Age','FamilySize','IsAlone','Embarked','Title'])]
    selected_cols = sorted(set(selected_cols + baseline_keep))
for c in selected_cols:
    if c not in X.columns:
        print(f'Warning: selected feature {c} not in training data columns.')
print(f'Selected {len(selected_cols)} features for RF (EDA-informed).')

X_train_sel = X_train[selected_cols].copy()
X_valid_sel = X_valid[selected_cols].copy()

# EDA-informed RF configuration (guided ranges from EDA)
rf_eda = RandomForestClassifier(
    n_estimators=randomSearchCV.best_params_['n_estimators'],
    max_depth=randomSearchCV.best_params_['max_depth'],
    min_samples_split=randomSearchCV.best_params_['min_samples_split'],
    min_samples_leaf=randomSearchCV.best_params_['min_samples_leaf'],
    max_features=randomSearchCV.best_params_['max_features'],
    class_weight=randomSearchCV.best_params_['class_weight'],
    bootstrap=randomSearchCV.best_params_['bootstrap'],
    oob_score=randomSearchCV.best_params_['oob_score'],
    n_jobs=-1,
    random_state=SEED
)
rf_eda.fit(X_train_sel, y_train)

pred = rf_eda.predict(X_valid_sel)
proba = rf_eda.predict_proba(X_valid_sel)[:,1]
acc = accuracy_score(y_valid, pred)
auc = roc_auc_score(y_valid, proba)
oob = rf_eda.oob_score_ if rf_eda.oob_score else np.nan
print({'mi_randomForest_accuracy': round(acc,4), 'mi_randomForest_roc_auc': round(auc,4)})
print('\nClassification report (EDA RF):\n', classification_report(y_valid, pred, digits=3))
print('\nConfusion matrix (EDA RF):\n', confusion_matrix(y_valid, pred))

plot_feature_importances(rf_eda, X_train_sel.columns, top_n=20, fname=f'{PIC_PATH}/mi_best25_rf_feature_importances_top20.png')

# Persist model and feature list
os.makedirs(PROCESS_PATH, exist_ok=True)
joblib.dump({'model': rf_eda, 'features': selected_cols}, f"{MODEL_PATH}/mi_randomForest_{TOP_K}_{timestamp}_features.pkl")
print(f"Saved EDA-tuned model to {MODEL_PATH}/mi_randomForest_{TOP_K}_{timestamp}_features.pkl")

Selected 17 features for RF (EDA-informed).


## EDA‑informed model with top‑K mutual information features

Here we incorporate domain‑driven feature selection signals from the previous EDA. The file `processed/eda_mutual_information_top30.csv` ranks features by mutual information with the target, capturing nonlinear and non‑monotonic associations that correlation alone may miss. We choose `TOP_K = 25` features and intersect them with the columns present in `X` to guard against name drift between preprocessing and modeling (e.g., one‑hot column naming differences). If the overlap is unexpectedly small, we apply a conservative fallback: keep a curated set of high‑signal patterns (Sex, Pclass, Fare, Age, FamilySize, IsAlone, Embarked, Title) by matching substrings across one‑hot columns. This ensures the model remains trainable and comparable even when the MI list and matrix diverge slightly.

We then construct `X_train_sel`/`X_valid_sel` using the same 80/20 split indices created earlier, but restricted to the selected columns. To isolate the effect of feature selection, we initialize a Random Forest with the best hyperparameters discovered by the randomized search. This allows a fair comparison: the only difference is the feature set. We report accuracy and ROC AUC on the validation subset, print a concise classification report and confusion matrix for clarity, and save an importance plot restricted to the selected features. Finally, we persist a versioned artifact to `MODEL_PATH` that includes both the trained estimator and the exact list of selected features.

Caveats and tips: MI ranking was computed from training data and thus reflects its distribution; if the data shifts, rerun EDA to refresh the list. Feature selection may reduce variance and improve interpretability, but if the baseline already generalizes well, gains can be modest. Downstream notebooks must strictly align to this saved feature list during inference to avoid column order or presence mismatches.

In [ ]:
print(timestamp)

20251102-032143
